### Checking the performance of CSV v/s parquet formats

In [ ]:
import pandas as pd
from pathlib import Path

DATA_PATH_CSV = Path("../data/processed/equity/master/nse_master_2016_2026.csv")
df = pd.read_csv(DATA_PATH_CSV)
df = df.sort_values(["symbol","trade_date"])
df

In [ ]:
import pandas as pd
from pathlib import Path

DATA_PATH = Path("../data/processed/equity/parquet/nse_master_2016_2026.parquet")
df = pd.read_parquet(DATA_PATH)
df = df.sort_values(["symbol","trade_date"])
df

<b>Note:</b><i>From above example we can see the difference between csv & parquet. That, <b>Parquet is more faster than csv</b></i>

## Daily Return

In [ ]:
df["return_1d"] = df.groupby("symbol")["close"].pct_change()
df

## Short Momentum

In [ ]:
df["return_5d"] = df.groupby("symbol")["close"].pct_change(5)
df

## Medium Momentum

In [ ]:
df["return_20d"] = df.groupby("symbol")["close"].pct_change(20)
df

## Rolling Volatility

In [ ]:
df["volatility_20d"] = (
    df.groupby("symbol")["return_1d"]
      .rolling(20)
      .std()
      .reset_index(level=0, drop=True)
)
df

## Liquidity Features

### Traded Value

In [ ]:
df["traded_value"] = df["close"] * df["tottrdqty"]
df

### Rolling liquidity

In [ ]:
df["avg_value_20d"] = (
    df.groupby("symbol")["traded_value"]
      .rolling(20)
      .mean()
      .reset_index(level=0, drop=True)
)
df

## Applying Liquidity Filters

In [ ]:
LIQUIDITY_THRESHOLD = 5e7

(₹5 crore average daily traded value)

### Filter dataset:clean tradable universe

In [ ]:
df = df[df["avg_value_20d"] > LIQUIDITY_THRESHOLD]
df = df[df["close"] > 20]

### Data after Liquidity Filters

In [ ]:
df

## Relative Strength vs Index

Load index features

In [ ]:
path = Path("../data/features/index")

files = sorted(path.glob("index_features_*.parquet"))

index_df = pd.concat(
    [pd.read_parquet(f) for f in files],
    ignore_index=True
)

index_df

Merging equity & index features

In [ ]:
df = df.merge(
    index_df[["trade_date","return_1d"]],
    on="trade_date",
    suffixes=("","_index")
)

Merged Data

In [ ]:
df

### Compute relative strength

In [ ]:
df["relative_strength"] = df["return_1d"] - df["return_1d_index"]
df

## Cross-Sectional Ranking

daily ranking across stocks

In [ ]:
df["rank_momentum"] = (
    df.groupby("trade_date")["return_20d"]
      .rank(ascending=False)
)

In [ ]:
df

Normalize ranking

In [ ]:
df["rank_pct"] = (
    df.groupby("trade_date")["return_20d"]
      .rank(pct=True)
)

In [ ]:
df

Normally ranking: Pandas converts ranks into percentiles</br>
Percentile Rank = Rank / Total Stocks</br>
| Stock   | return_20d | rank | rank_pct |
| ------- | ---------- | ---- | -------- |
| Stock A | 0.50       | 1    | 0.001    |
| Stock B | 0.45       | 50   | 0.05     |
| Stock C | 0.10       | 500  | 0.50     |
| Stock D | -0.30      | 1000 | 1.00     |
<hr>
0.05 → top 5% momentum stock</br>
0.50 → average stock</br>
1.00 → worst momentum stock

## Save Features (Partitioned Parquet)

In [ ]:
df["year"] = df["trade_date"].dt.year
df

In [ ]:
output_path = Path("../data/features/equity")
output_path.mkdir(parents=True, exist_ok=True)

years = df["year"].unique()

for year in years:
    
    df_year = df[df["year"] == year]
    
    file_path = output_path / f"equity_features_{year}.parquet"
    
    if not file_path.exists():
        df_year.to_parquet(file_path, index=False)